# Python CSV Files

> 📘 **Python Mastery** · Module 06 — File Handling · Lesson 4/4

The world's data ships as **CSV** — spreadsheets, bank statements, Kaggle datasets, ML training sets. CSV looks like plain text (it is!), but it has enough edge cases — commas inside values, Windows line endings, headers — that Python ships a dedicated `csv` module. Master it here and Module 11's pandas will feel like a magic wand waved over concepts you already own.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- Describe the structure of a CSV file (header row, records, delimiter).
- Read CSVs with `csv.reader`, treating the header with `next()`.
- Write CSVs with `csv.writer`, `writerow()` and `writerows()`.
- Explain **why** `newline=""` matters when opening CSV files on Windows.
- Use `DictReader` / `DictWriter` to treat rows as dictionaries instead of lists.
- Compute simple statistics (per-row and per-column averages) from a real dataset with pure stdlib code.

## 1. What Exactly Is a CSV?

**CSV = Comma-Separated Values.** Each line is a *record*; within a line, commas split the *fields*. The first line is conventionally a **header** naming each column. It is the lingua franca of spreadsheets — Excel, Google Sheets, and pandas all open it natively — precisely because it is just text you could type in Notepad.

```text
Name,Math,Science,English
Sarah,88,92,85
Tanvir,75,81,79
```

Because it is only text, we can *create* our practice dataset with skills from Lesson 2 — no spreadsheet required. Let's build a class gradebook for five students.

In [1]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

raw_csv = """Name,Math,Science,English
Sarah,88,92,85
Tanvir,75,81,79
Amina,95,89,94
Rafi,67,74,70
Nusrat,82,78,90
"""

students_path = Path("sample_data", "students.csv")
students_path.write_text(raw_csv, encoding="utf-8")

print(students_path.read_text(encoding="utf-8"))
print("-> That is ALL a CSV is: text with commas.")

Name,Math,Science,English
Sarah,88,92,85
Tanvir,75,81,79
Amina,95,89,94
Rafi,67,74,70
Nusrat,82,78,90

-> That is ALL a CSV is: text with commas.


## 2. Reading with `csv.reader` — Rows as Lists

`csv.reader(f)` wraps an open file and yields one **list of strings per row**. Why not just `line.split(",")`? Because real CSVs hide surprises: quoted fields containing commas (`"Dhaka, Bangladesh"`), embedded quotes, multi-line cells. The module handles all of that; naive splitting does not.

**Syntax:**

```python
import csv

with open("data.csv", newline="", encoding="utf-8") as f:
    for row in csv.reader(f):
        print(row)          # row is a LIST of strings
```

(The `newline=""` is required — Section 5 explains.)

**Example:**

In [2]:
import csv

with open("sample_data/students.csv", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row, "<-", type(row).__name__)

['Name', 'Math', 'Science', 'English'] <- list
['Sarah', '88', '92', '85'] <- list
['Tanvir', '75', '81', '79'] <- list
['Amina', '95', '89', '94'] <- list
['Rafi', '67', '74', '70'] <- list
['Nusrat', '82', '78', '90'] <- list


Every row arrives as a list of **strings** — even marks like `88` are text until you convert them with `int()`. The first loop iteration is really the header wearing a row costume. The clean idiom pulls it off first with `next()`, which advances any iterator one step:

**Syntax:**

```python
reader = csv.reader(f)
header = next(reader)       # consume the header row
first_row = next(reader)    # ...and optionally peek at the first record
```

**Example:**

In [3]:
import csv

with open("sample_data/students.csv", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)               # pull off the header...
    print("Header:", header)
    print("Columns:", len(header), "->", header[0], "|", header[2])

    first_student = next(reader)        # ...then the data starts clean
    print("First student:", first_student)

Header: ['Name', 'Math', 'Science', 'English']
Columns: 4 -> Name | Science
First student: ['Sarah', '88', '92', '85']


## 3. Writing with `csv.writer`

The mirror image: `csv.writer(f)` converts your Python lists into properly escaped CSV text. `writerow(row_list)` emits one row; `writerows(list_of_rows)` emits them all. Crucially, the writer **quotes fields for you** — a value like `"Dhaka, Bangladesh"` gets wrapped in quotes automatically, which is exactly the bug manual string-building produces.

**Syntax:**

```python
import csv

with open("out.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["col_a", "col_b"])   # header
    writer.writerow([1, "hello"])
    writer.writerows([[2, "world"], [3, "!"]])
```

**Example:**

In [4]:
import csv
from pathlib import Path

rows = [
    ["Subject", "Credits", "Grade"],
    ["Math", 4, "A"],
    ["Science", 3, "A-"],
    ["History", 2, "B+"],
]

with open("sample_data/subjects.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(rows[0])        # header, one row
    writer.writerows(rows[1:])      # data, many rows

print(Path("sample_data/subjects.csv").read_text(encoding="utf-8"))

# The escaping superpower: commas INSIDE values are handled.
with open("sample_data/cities.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(
        [["City", "Country"], ["Chattogram", "Bangladesh"], ["Kolkata", "India, West Bengal"]]
    )
print(Path("sample_data/cities.csv").read_text(encoding="utf-8"))

Subject,Credits,Grade
Math,4,A
Science,3,A-
History,2,B+

City,Country
Chattogram,Bangladesh
Kolkata,"India, West Bengal"



Notice `India, West Bengal` came out wrapped in quotes — the writer protected that comma. Any spreadsheet parses it back as a single field.

## 4. Why `newline=""` Is Required (Especially on Windows)

CSV is defined to end rows with `\r\n` (a holdover from typewriter days: carriage-return + line-feed, kept for spreadsheet compatibility). Meanwhile, opening a file in text mode on Windows normally *translates* `\n` into `\r\n` on the way out. Put those two behaviours together and each row ends up with `\r\r\n` — Excel renders that as a **blank row between every data row**. Passing `newline=""` tells `open()`: *"don't translate anything — let the csv module control line endings."* Reading needs it too, for symmetric reasons.

> 🔍 **Under the Hood:** `newline=""` disables Python's universal-newline translation in both directions. The `csv` module then writes exactly `\r\n` per row (per the CSV RFC) and parses any of `\r\n`, `\n`, `\r` on input. One argument, and your files look identical in Excel on Windows, LibreOffice on Linux, and pandas everywhere.

In [5]:
import csv
from pathlib import Path

rows = [["City", "Population"], ["Dhaka", 22479000], ["Chattogram", 5441000]]

# WITHOUT newline="" : Windows inserts extra \r -> blank rows in Excel.
with open("sample_data/cities_wrong.csv", "w", encoding="utf-8") as f:
    csv.writer(f).writerows(rows)
print("wrong bytes:", repr(Path("sample_data/cities_wrong.csv").read_bytes()))

# WITH newline="" : clean, standard CSV.
with open("sample_data/cities_right.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(rows)
print("right bytes:", repr(Path("sample_data/cities_right.csv").read_bytes()))

wrong bytes: b'City,Population\r\r\nDhaka,22479000\r\r\nChattogram,5441000\r\r\n'
right bytes: b'City,Population\r\nDhaka,22479000\r\nChattogram,5441000\r\n'


## 5. `DictReader` — Rows as Dictionaries

Indexing rows by position (`row[2]`) is brittle: insert one column and every index shifts silently. `csv.DictReader` uses the header to give you each row as a **dictionary** keyed by column name — `row["Science"]` reads like what it means. For anything beyond toy scripts, this is the way.

**Syntax:**

```python
with open("data.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        row["column_name"]      # values are strings!
```

**Example:**

In [6]:
import csv

with open("sample_data/students.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        # Keys come straight from the header row.
        print(f"{row['Name']:<8} Math={row['Math']} (type: {type(row['Math']).__name__})")

Sarah    Math=88 (type: str)
Tanvir   Math=75 (type: str)
Amina    Math=95 (type: str)
Rafi     Math=67 (type: str)
Nusrat   Math=82 (type: str)


In [7]:
import csv

# DictReader consumed the header for us - prove the keys match it.
with open("sample_data/students.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("fieldnames:", reader.fieldnames)
    row = next(reader)
    print("first row :", dict(row))

fieldnames: ['Name', 'Math', 'Science', 'English']
first row : {'Name': 'Sarah', 'Math': '88', 'Science': '92', 'English': '85'}


## 6. `DictWriter` — Writing Dictionaries Back Out

`csv.DictWriter` takes each row as a dict, but you must tell it the column order up front via `fieldnames=` (dicts themselves have no guaranteed order for output). Call `writeheader()` first, then `writerow()`/`writerows()` as usual. Extra safety net: `extrasaction="ignore"` silently drops dict keys you did not declare.

**Syntax:**

```python
writer = csv.DictWriter(f, fieldnames=["Name", "Score"])
writer.writeheader()
writer.writerow({"Name": "Sarah", "Score": 92})
```

**Example:** turn the raw gradebook into a computed summary file.

In [8]:
import csv
from statistics import mean

# Compute each student's average, ready to export.
summary = []
with open("sample_data/students.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        marks = [int(row[c]) for c in ("Math", "Science", "English")]
        summary.append({"Student": row["Name"], "Average": round(mean(marks), 1)})

with open("sample_data/student_averages.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["Student", "Average"])
    writer.writeheader()
    writer.writerows(summary)

from pathlib import Path
print(Path("sample_data/student_averages.csv").read_text(encoding="utf-8"))

Student,Average
Sarah,88.3
Tanvir,78.3
Amina,92.7
Rafi,70.3
Nusrat,83.3



## 7. Realistic Analysis: Stats Straight from the File

Everything so far combines into the classic beginner-data-analysis script: read a dataset, compute **per-row** stats (each student's average) and **per-column** stats (each subject's difficulty), find the champion — all with pure standard library, no libraries installed.

In [9]:
import csv
from statistics import mean

with open("sample_data/students.csv", newline="", encoding="utf-8") as f:
    students = list(csv.DictReader(f))

subjects = ["Math", "Science", "English"]
print(len(students), "students loaded.\n")

# Per-ROW statistic: each student's average across subjects.
print("Per-student average")
averages = {}
for s in students:
    avg = mean(int(s[sub]) for sub in subjects)
    averages[s["Name"]] = round(avg, 1)
    print(f"  {s['Name']:<8} {avg:5.1f}")

# Per-COLUMN statistic: each subject's class average.
print("\nPer-subject average (lower = harder subject)")
for sub in subjects:
    col = [int(s[sub]) for s in students]
    print(f"  {sub:<8} {mean(col):5.1f}")

top_student = max(averages, key=averages.get)
hardest = min(subjects, key=lambda sub: mean(int(s[sub]) for s in students))
print(f"\nTop student : {top_student} ({averages[top_student]})")
print(f"Hardest exam: {hardest}")

5 students loaded.

Per-student average
  Sarah     88.3
  Tanvir    78.3
  Amina     92.7
  Rafi      70.3
  Nusrat    83.3

Per-subject average (lower = harder subject)
  Math      81.4
  Science   82.8
  English   83.6

Top student : Amina (92.7)
Hardest exam: Math


## 8. Other Delimiters: Semicolons and Tabs

Comma is the default, not a law. European locales often use `;` (their comma is the decimal mark), and tab-separated values (**TSV**) paste beautifully into spreadsheets. Both `csv.reader` and `csv.writer` accept `delimiter=...`; the rest of the machinery is unchanged.

**Syntax:**

```python
csv.reader(f, delimiter=";")
csv.writer(f, delimiter="\t")     # "\t" is the tab character
```

**Example:**

In [10]:
import csv
from pathlib import Path

scores = [["name", "score"], ["Sarah", 88], ["Tanvir", 75]]

# Semicolon-separated (common in European exports).
with open("sample_data/scores_semicolon.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f, delimiter=";").writerows(scores)

# Tab-separated (.tsv).
with open("sample_data/scores.tsv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f, delimiter="\t").writerows(scores)

print(repr(Path("sample_data/scores_semicolon.csv").read_text(encoding="utf-8")))
print(repr(Path("sample_data/scores.tsv").read_text(encoding="utf-8")))

# Reading respects the SAME delimiter option.
with open("sample_data/scores_semicolon.csv", newline="", encoding="utf-8") as f:
    for row in csv.reader(f, delimiter=";"):
        print(row)

'name;score\nSarah;88\nTanvir;75\n'
'name\tscore\nSarah\t88\nTanvir\t75\n'
['name', 'score']
['Sarah', '88']
['Tanvir', '75']


### Teaser: pandas does all of this in one line

Once you reach Module 11, the section above collapses to:

```python
import pandas as pd

df = pd.read_csv("sample_data/students.csv")   # header, types, everything
df.mean(numeric_only=True)                     # per-column stats, done
```

pandas builds directly on these fundamentals — knowing what `read_csv` is doing for you (delimiters, quoting, headers, dtypes) is exactly what separates people who *use* pandas from people who can *debug* it.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---------|---------|-----|
| Opening CSVs without `newline=""` | Blank rows between records on Windows | Always `open(p, newline="", encoding="utf-8")` |
| Building rows manually with `",".join(parts)` | A value containing a comma corrupts the file | Use `csv.writer` — it quotes automatically |
| Parsing with `line.split(",")` | Breaks on quoted commas, escaped quotes | Use `csv.reader` |
| Forgetting values arrive as strings | `sum(row["Math"])` fails mysteriously | Convert explicitly: `int(row["Math"])` |
| Looping `csv.reader` including the header | Header sneaks into calculations | Pull it out with `next(reader)` or use `DictReader` |
| Reading a `;`-file with default comma | One giant field per row | Pass the matching `delimiter=` |

## 💡 Best Practices & Pro Tips

- Make `open(path, "w", newline="", encoding="utf-8")` muscle memory — the trio fixes Windows endings *and* international characters in one go.
- Prefer `DictReader`/`DictWriter` for anything with more than ~3 columns: named access survives schema changes.
- Validate while loading: check `reader.fieldnames` against expectations before trusting the data.
- For datasets beyond a few thousand rows, or when types matter, jump to pandas (Module 11) — the `csv` module is the foundation, not the finish line.
- **AI-engineering relevance:** datasets arrive as CSV more than any other format — features, labels, evaluation results, prediction dumps. Before a single tensor is created, someone must parse, validate and summarise these files; the per-column statistics you just wrote by hand are the same ones `pandas.describe()` automates later. Debugging a "bad line" in a 10 GB training CSV is a rite of passage this lesson prepares you for.

## 📌 Summary

| Tool | What it does | Example |
|------|--------------|---------|
| `csv.reader(f)` | Yields each row as a list of strings | `for row in csv.reader(f):` |
| `next(reader)` | Consumes the header row | `header = next(reader)` |
| `csv.writer(f)` | Writes lists as escaped CSV rows | `writer.writerow(["a", 1])` |
| `.writerow()` / `.writerows()` | One row / many rows | `writer.writerows(data)` |
| `csv.DictReader(f)` | Rows as dicts keyed by header | `row["Name"]` |
| `csv.DictWriter(f, fieldnames=...)` | Dicts → columns in your chosen order | `writer.writeheader()` |
| `delimiter=";"` / `"\t"` | Non-comma separators | `csv.reader(f, delimiter="\t")` |
| `newline=""` | Stops Windows line-ending corruption | `open(p, "w", newline="")` |

Key takeaways:

- CSV is plain text with conventions — which is why the `csv` module, not string slicing, owns the parsing.
- Lists are positional; dicts are descriptive: graduate from `reader` to `DictReader` as tables grow.
- `newline=""` is mandatory, not optional polish, especially on Windows.
- Per-row and per-column statistics are the same loop pointed in different directions — and pandas (Module 11) automates exactly this.

## 🔗 Next Lesson

Real-world data breaks things: missing files, malformed rows, zero divisions. Module 07 teaches your programs to absorb the blow — starting with try/except in [`../../07_Error_Handling/01_Try_Except/notes.ipynb`](../../07_Error_Handling/01_Try_Except/notes.ipynb).